# Week 09 Day 07 Learning Notebook

This day notebook is fully executable and aligned to the lesson content.

In [1]:
from datetime import date
import warnings
import numpy as np
import pandas as pd

try:
    import yfinance as yf
except Exception:
    yf = None

try:
    from pandas_datareader import data as pdr
except Exception:
    pdr = None

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.6f}'.format

def _synthetic_price_frame(tickers, start='2018-01-01', periods=900, seed=7):
    idx = pd.bdate_range(start, periods=periods)
    out = pd.DataFrame(index=idx)
    rng = np.random.default_rng(seed)
    for i, ticker in enumerate(tickers):
        drift = 0.0002 + 0.00005 * (i + 1)
        vol = 0.010 + 0.002 * i
        rets = rng.normal(drift, vol, size=len(idx))
        out[ticker] = 100.0 * np.exp(np.cumsum(rets))
    return out

def load_market_prices(tickers, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    tickers = list(tickers)
    if yf is None:
        warnings.warn('yfinance unavailable, using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

    try:
        raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
        if isinstance(raw.columns, pd.MultiIndex):
            if 'Close' in raw.columns.get_level_values(0):
                close = raw['Close']
            else:
                close = raw.xs('Close', axis=1, level=0, drop_level=True)
        else:
            close = raw.rename(columns={raw.columns[0]: tickers[0]})
        close = close.reindex(columns=tickers)
        close = close.dropna(how='all').ffill().dropna()
        if close.empty:
            raise ValueError('empty market data from Yahoo Finance')
        return close
    except Exception as exc:
        warnings.warn(f'Yahoo download failed ({exc}); using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

def load_fred_series(series_id, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    if pdr is None:
        warnings.warn('pandas_datareader unavailable, using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

    try:
        ser = pdr.DataReader(series_id, 'fred', start, end)[series_id]
        ser = ser.ffill().dropna()
        if ser.empty:
            raise ValueError('empty FRED series')
        return ser
    except Exception as exc:
        warnings.warn(f'FRED download failed ({exc}); using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

# Week 09 Day 07: Portfolio Project

## Study Duration
- Planned effort: 6-10 hours/day
- Required minimum: 6 hours for implementation, validation, and communication drills.

## Continuity and Handoff
- Previous checkpoint: Week 09 Day 06: Revision Sprint
- Previous lesson file: content/week-09/day-06.md
- Today's deliverable: Time-series baseline forecasting notebook
- Next handoff target: Week 10 Day 01: Volatility stylized facts
- Next lesson file: content/week-10/day-01.md

## Project Title
Time-series baseline forecasting notebook

## Problem Statement
Build and compare baseline time-series models with proper diagnostics.

## Data Sources
- Synthetic AR data
- Market index returns
- Volatility proxy

## Implementation Steps
1. Prepare and transform series
2. Fit baseline models
3. Run residual diagnostics
4. Evaluate forecasts
5. Document practical takeaway

## Evaluation Metrics
- MAE
- RMSE
- Residual whiteness
- Economic usefulness

## Execution Standard
- [ ] Notebook/script runs from clean start without hidden state
- [ ] Outputs include at least one diagnostic table and one chart
- [ ] One explicit risk guardrail and fallback action are documented

## Deliverables
- Notebook or script output
- One-page summary memo
- Tracker update with completion and lessons learned


## Runnable Day Example
Run this example for Week 09 Day 07, inspect outputs, then complete the quiz.

In [2]:
# Project day demo: mini portfolio report with trade recommendation
np.random.seed(907)
assets = ['SPY', 'QQQ', 'TLT', 'GLD']
prices = load_market_prices(assets, start='2019-01-01')
returns = prices.pct_change().dropna()

annual_return = returns.mean() * 252
annual_vol = returns.std() * np.sqrt(252)
score = (annual_return - 0.03) / annual_vol.replace(0, np.nan)

report = pd.DataFrame({
    'annual_return': annual_return,
    'annual_vol': annual_vol,
    'sharpe_proxy': score,
}).sort_values('sharpe_proxy', ascending=False)

top_asset = report.index[0]
print('Project summary:')
print(report.round(4))
print(f"\nSuggested focus asset for follow-up research: {top_asset}")


Project summary:
        annual_return  annual_vol  sharpe_proxy
Ticker                                         
GLD          0.194100    0.172700      0.950000
QQQ          0.232200    0.240200      0.841900
SPY          0.177800    0.196000      0.754000
TLT         -0.005100    0.162600     -0.215600

Suggested focus asset for follow-up research: GLD


## ReAct Verification Cell
Use this execution cell to validate one trade decision with an explicit risk guardrail.

In [3]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(9907)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Portfolio Project',
    'week': 9,
    'day': 7,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Portfolio Project',
 'week': 9,
 'day': 7,
 'observe_annual_return': 0.1492338409257925,
 'observe_annual_vol': 0.20501283741536433,
 'reason_sharpe_proxy': 0.5815920721306828,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 09 Day 07 Quiz

Topic: **Portfolio Project**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Which formula from today's lesson directly drives a trade decision, and what does each symbol mean?
2. Risk question: Using one real ticker from today's example, what hard guardrail would you enforce before live deployment?
3. Communication question: In one minute, explain why this topic matters for production trading systems.

Scoring: full credit requires notation correctness, one numeric example, and one explicit risk control.

In [4]:
# Quiz model answers (auto-generated)
price_t_minus_1 = 116.000
price_t = 117.334
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Interview Question 1 (model answer):')
print('  I would use simple return to convert price moves into decision-ready percentages under volatility-clustering window.')
print('  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce this guardrail before deployment:')
print('  de-risk when realized volatility exceeds model regime.')

print('\nInterview Question 3 (model answer):')
print('  Topic:', 'Portfolio Project')
print('  This matters because production systems need reproducible metrics, explicit controls,')
print('  and a fallback decision path when stress conditions invalidate baseline assumptions.')

print('\nNumeric verification:')
print('  simple_return_expected =', 0.011500)
print('  gross_return_expected  =', 1.011500)


Interview Question 1 (model answer):
  I would use simple return to convert price moves into decision-ready percentages under volatility-clustering window.
  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)
  P_(t-1): 116.0
  P_t    : 117.334
  r_t    : 0.0115 => 1.15%
  1+r_t  : 1.0115

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce this guardrail before deployment:
  de-risk when realized volatility exceeds model regime.

Interview Question 3 (model answer):
  Topic: Portfolio Project
  This matters because production systems need reproducible metrics, explicit controls,
  and a fallback decision path when stress conditions invalidate baseline assumptions.

Numeric verification:
  simple_return_expected = 0.0115
  gross_return_expected  = 1.0115
